In [ ]:
from openai import OpenAI
import gradio as gr
from pydantic import BaseModel, Field

c:\Users\LENOVO\Documents\projects\llm_mini_projects\prisoners-dilemma\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
# Defining local ollama variables

DEEPSEEK_R1_1_5B = "deepseek-r1:1.5b"
LLAMA_3_2_1B = "llama3.2:1b"
GEMMA_3_270M = "gemma3:270m"

In [12]:
# Validate ollama running
import requests
requests.get("http://localhost:11434").content

b'Ollama is running'

In [13]:
# Ollama instance
ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [26]:
# Defining the game for 3 agents (not specifically meaning AI Agents - Agents as in anything that can perceive the environment and act on it)

PRISONERS_DILEMMA_TRIO_PART_SYSTEM_PROMPT = """
## Prisoner's Dilemma for three suspects

The 3-player Prisoner's Dilemma extends the classic two-player game to three suspects - Adam, Eve and Bob who must independently choose whether to Cooperate (stay silent) or Defect (confess / betray).


## The Setup & Rules

Each player acts in self-interest without knowing the others' choices. Based on a standard jail-time payoff structure (where fewer months in prison is better):
• All three cooperate (CCC): Each player gets 4 months in jail. (Best collective outcome)
• All three defect (DDD): Each player gets 8 months in jail.
• Two defect, one cooperates (e.g., DDC, DCD, CDD):
	• The two defectors get 6 months each.
	• The single cooperator gets 12 months.
• One defects, two cooperate (e.g., CCD, CDC, DCC):
	• The single defector walks free (0 months).
	• The two cooperators get 12 months each.

Your response should be in a clear markdown format without unnecessary backticks. 

The number of total rounds the same three players will play is 5. Each round begins fresh, but with each player aware of the response made by the other 2. 

The response format should only be in one phrase - "Stay Silent" or "Betray". 
"""

In [27]:
adam_messages = ["Hi, I'm Adam !"]
eve_messages = ["Hi, I'm Eve !"]
bob_messages = ["Hi, I'm Bob !"]

In [ ]:
def call_adam() -> str:
    adam_system_message = PRISONERS_DILEMMA_TRIO_PART_SYSTEM_PROMPT + "\n\nIn this scenario, you are Adam."
    messages = [
        {
            "role": "system",
            "content": adam_system_message
        }
    ]
    for eve_message, bob_message in zip(eve_messages, bob_messages):
        messages.append({
            "role": "user",
            "content": f"""
                Eve says: {eve_message},\n
                Bob says: {bob_message}
            """
        })
    response = ollama.chat.completions.create(
        model=DEEPSEEK_R1_1_5B,
        messages=messages,
    )
    if response.choices and response.choices[0].message:
        return response.choices[0].message.content 
    else:
        raise ValueError("No response from Deepseek received")

In [24]:
def call_eve() -> str:
    eve_system_message = PRISONERS_DILEMMA_TRIO_PART_SYSTEM_PROMPT + "\n\nIn this scenario, you are Eve."
    messages = [
        {
            "role": "system",
            "content": eve_system_message
        }
    ]
    for adam_message, bob_message in zip(adam_messages, bob_messages):
        messages.append({
            "role": "user",
            "content": f"""
                Adam says: {adam_message},\n
                Bob says: {bob_message}
            """
        })
    response = ollama.chat.completions.create(
        model=LLAMA_3_2_1B,
        messages=messages
    )
    if response.choices and response.choices[0].message:
        return response.choices[0].message.content 
    else:
        raise ValueError("No response from Llama received")

In [25]:
def call_bob() -> str:
    bob_system_message = PRISONERS_DILEMMA_TRIO_PART_SYSTEM_PROMPT + "\n\nIn this scenario, you are Bob."
    messages = [
        {
            "role": "system",
            "content": bob_system_message
        }
    ]
    for adam_message, eve_message in zip(adam_messages, eve_messages):
        messages.append({
            "role": "user",
            "content": f"""
                Adam says: {adam_message},\n
                Eve says: {eve_message}
            """
        })
    response = ollama.chat.completions.create(
        model=GEMMA_3_270M,
        messages=messages
    )
    if response.choices and response.choices[0].message:
        return response.choices[0].message.content 
    else:
        raise ValueError("No response from Gemma received")